# Phase 4 Demo — Query Enrichment & Entity Graph Search

## What This Notebook Shows

This interactive notebook demonstrates how our advanced image retrieval system processes text queries:

1. **Simple text query** → A user types a description (e.g., "a girl playing violin on stage")
2. **Finding example images (seeds)** → The system finds initial matching images using CLIP
3. **Discovering related concepts** → From those examples, it identifies important entities/concepts
4. **Query enrichment** → The original query is enhanced with "Related: [concept1, concept2, ...]"
5. **Graph-based search** → A knowledge graph of concepts helps find more relevant images
6. **Comparison** → See how results differ between:
   - **CLIP-only** (traditional semantic search)
   - **CLIP + Knowledge Graph (KG)** (concept-enriched search)

### Interactive Experience

You can type your own queries and watch the full pipeline in action!

---

**⚠️ Before Running:**

Please ensure you have:
- **Activated your virtual environment** (e.g., `venv\Scripts\Activate.ps1` on Windows or `source venv/bin/activate` on Unix)
- **Selected it as the Python kernel** in VS Code (use the kernel picker in the top-right corner)
- All project dependencies installed (`pip install -r requirements.txt`)
- Phase 4 data artifacts available (entity graph, embeddings, indices)

## Setup & Imports

In this step we:
- Import necessary libraries
- Load Phase 4 configuration from `configs/entity_graph.yaml`
- Prepare the components: text encoder (CLIP), entity graph, and metadata

In [1]:
# Clone your GitHub repository
!git clone -b phase4-entity --single-branch https://github.com/vinhhna/hybrid_multimodal_retrieval.git
%cd /kaggle/working/hybrid_multimodal_retrieval

!pip install --prefer-binary --no-build-isolation \
  torch-geometric torch-scatter torch-sparse \
  -f https://data.pyg.org/whl/torch-2.6.0+cu124.html


!grep -Ev '^(torch(|vision|audio)|torch-geometric|torch-scatter|torch-sparse)\b' requirements.txt > requirements-core.txt
!pip install --prefer-binary --no-build-isolation -r requirements-core.txt
!python -m spacy download en_core_web_sm

# Install project in development mode
!pip install -e .

import sys
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

fatal: destination path 'hybrid_multimodal_retrieval' already exists and is not an empty directory.
/kaggle/working/hybrid_multimodal_retrieval
Looking in links: https://data.pyg.org/whl/torch-2.6.0+cu124.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 102.5 MB/s eta 0:00:0000:010:01
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Obtaining file:///kaggle/working/hybrid_multimodal_retrieval
  Preparing metadata (setup.py) ... done
  Attempting uninstall: flickr30k
    Found existing installation: flickr30k 0.1.0
    Uninstalling flickr30k-0.1.0:
 

In [ ]:
import os
import json
from pathlib import Path
from typing import List, Tuple, Dict, Any

import torch
import numpy as np
from IPython.display import display, Image as IPyImage, HTML
from PIL import Image as PILImage

# Project imports
from src.graph.config import (
    load_entity_graph_config,
    get_entity_graph_config,
    get_query_enrichment_config,
    get_graph_search_config,
    get_fusion_config,
)
from src.graph.graph_search import (
    enrich_query,
    graph_search,
    image_scores_dict,
)
from src.graph.build_entity_graph import load_entity_graph
from src.retrieval.bi_encoder import BiEncoder
from src.retrieval.hybrid_search import HybridSearchEngine

# Define global IMAGE_ROOT for Kaggle
IMAGE_ROOT = Path("/kaggle/input/flickr30k/data/images")
print("Using IMAGE_ROOT:", IMAGE_ROOT)

CFG_PATH = "configs/entity_graph.yaml"

raw_cfg = load_entity_graph_config(CFG_PATH)

# ------------------------------------------------------------------
# KAGGLE-SPECIFIC OVERRIDE:
# Treat all 'data/...' paths as relative to /kaggle/input/flickr30k
# ------------------------------------------------------------------
DATA_ROOT = Path("/kaggle/input/flickr30k")

def remap_entity_graph_paths(cfg: dict, data_root: Path) -> dict:
    """
    Remap all entity_graph paths so that relative paths like
    'data/graph/entity_graph.pt' become
    '/kaggle/input/flickr30k/data/graph/entity_graph.pt'.
    """
    eg = cfg.get("entity_graph", {})
    path_keys = [
        "vocab_path",
        "context_path",
        "entity_embeddings_path",
        "entity_meta_path",
        "entity_graph_path",
    ]

    for key in path_keys:
        if key in eg:
            p = Path(eg[key])
            # If it's not absolute, make it relative to DATA_ROOT
            if not p.is_absolute():
                eg[key] = str(data_root / p)
    cfg["entity_graph"] = eg
    return cfg

raw_cfg = remap_entity_graph_paths(raw_cfg, DATA_ROOT)

entity_cfg = get_entity_graph_config(raw_cfg)
qe_cfg = get_query_enrichment_config(raw_cfg)
gs_cfg = get_graph_search_config(raw_cfg)
fusion_cfg = get_fusion_config(raw_cfg)

print("Using entity graph path:", entity_cfg["entity_graph_path"])


print("✓ Loaded Phase 4 configuration")
print(f"  • Entity graph path: {entity_cfg['entity_graph_path']}")
print(f"  • Query enrichment: K_seed_raw = {qe_cfg['K_seed_raw']}, M_enrich = {qe_cfg['M_enrich']}")
print(f"  • Graph search: H_max = {gs_cfg['H_max']}, decay = {gs_cfg['decay']}")
print(f"  • Fusion default mode: {fusion_cfg.get('default_mode')}")

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Using entity graph path: /kaggle/input/flickr30k/data/graph/entity_graph.pt
✓ Loaded Phase 4 configuration
  • Entity graph path: /kaggle/input/flickr30k/data/graph/entity_graph.pt
  • Query enrichment: K_seed_raw = 32, M_enrich = 8
  • Graph search: H_max = 2, decay = 0.85
  • Fusion default mode: full


## Load Models, Graph, and Metadata

Now we load:
1. **The text encoder (CLIP)** — To understand your query and each concept (entity)
2. **The entity graph** — A "map of concepts" showing how ideas connect to each other
3. **Metadata** — Information about which images and captions each concept appears in

In [3]:
# Determine device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Instantiate text/image encoder (CLIP)
print("\nLoading CLIP encoder...")
bi_encoder = BiEncoder(device=device)
print("✓ CLIP encoder loaded")

# Load required components for HybridSearchEngine
print("\nLoading search components...")

# Load cross-encoder (BLIP-2)
from src.retrieval.cross_encoder import CrossEncoder
cross_encoder = CrossEncoder(device=device)
print("✓ Cross-encoder (BLIP-2) loaded")

# Load FAISS index
from src.retrieval.faiss_index import FAISSIndex
image_index = FAISSIndex(embedding_dim=512)  # CLIP embedding dimension
image_index.load("/kaggle/input/flickr30k/data/indices/image_index.faiss")
print(f"✓ Image index loaded ({image_index.index.ntotal:,} vectors)")

# Load dataset
from src.flickr30k.dataset import Flickr30KDataset
dataset = Flickr30KDataset()
print(f"✓ Dataset loaded ({len(dataset):,} images)")

# Instantiate the full HybridSearchEngine
print("\nInitializing hybrid search engine...")
hybrid_engine = HybridSearchEngine(
    bi_encoder=bi_encoder,
    cross_encoder=cross_encoder,
    image_index=image_index,
    dataset=dataset,
    entity_graph=None,  # Will load separately
    phase4_cfg=raw_cfg,  # Pass Phase 4 config
)
print("✓ Hybrid search engine initialized")

Using device: cuda

Loading CLIP encoder...
Loading CLIP model: ViT-B-32 (openai)
Using device: cuda


/usr/local/lib/python3.11/dist-packages/open_clip/factory.py:388: UserWarning: These pretrained weights were trained with QuickGELU activation but the model config does not have that enabled. Consider using a model config with a "-quickgelu" suffix or enable with a flag.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✓ CLIP model loaded successfully
✓ CLIP encoder loaded

Loading search components...
Initializing BLIP-2 Cross-Encoder (Hugging Face)...
Loaded config from /kaggle/working/hybrid_multimodal_retrieval/configs/blip2_config.yaml
Using device: cuda
Loading BLIP-2 model from Hugging Face: Salesforce/blip2-opt-2.7b


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using FP16 mixed precision
  ✓ Yes token IDs: [10932, 4420, 9904, 3216]
  ✓ No token IDs: [2362, 117, 3084, 440]
✓ BLIP-2 Cross-Encoder initialized successfully
✓ Cross-encoder (BLIP-2) loaded
✓ Index loaded from: /kaggle/input/flickr30k/data/indices/image_index.faiss
  Vectors in index: 31,783
✓ Metadata loaded from: /kaggle/input/flickr30k/data/indices/image_index.json
✓ Image index loaded (31,783 vectors)
✓ Dataset loaded (0 images)

Initializing hybrid search engine...
Initializing Hybrid Search Engine...
✓ Hybrid Search Engine initialized
  Stage 1: CLIP (ViT-B-32)
  Stage 2: BLIP-2
  Image Index: 31,783 vectors
  Dataset: 0 images
  Config: k1=50, k2=10, batch_size=8, fusion=weighted
✓ Hybrid search engine initialized


/kaggle/working/hybrid_multimodal_retrieval/src/flickr30k/dataset.py:53: UserWarning: Captions file not found: data/results.csv
  warnings.warn(f"Captions file not found: {self.captions_file}")
/kaggle/working/hybrid_multimodal_retrieval/src/flickr30k/dataset.py:56: UserWarning: Images directory not found: data/images
  warnings.warn(f"Images directory not found: {self.images_dir}")


In [4]:
# Load the entity graph
graph_path = entity_cfg["entity_graph_path"]
print(f"Loading entity graph from: {graph_path}")
graph = load_entity_graph(graph_path)

# The graph is a PyTorch Geometric HeteroData object
# Extract statistics about the graph structure
num_entities = graph["entity"].x.size(0)
num_sem_edges = graph["entity", "sem", "entity"].edge_index.size(1) if ("entity", "sem", "entity") in graph.edge_types else 0
num_cooc_edges = graph["entity", "cooc", "entity"].edge_index.size(1) if ("entity", "cooc", "entity") in graph.edge_types else 0

print(f"✓ Entity graph loaded")
print(f"  • Entities: {num_entities}")
print(f"  • Semantic edges: {num_sem_edges}")
print(f"  • Co-occurrence edges: {num_cooc_edges}")

# Load entity context and vocabulary/metadata
print("\nLoading entity metadata...")
with open(entity_cfg["context_path"], "r") as f:
    entity_context: Dict[str, Any] = json.load(f)

with open(entity_cfg["vocab_path"], "r") as f:
    entity_vocab: Dict[str, Any] = json.load(f)

with open(entity_cfg["entity_meta_path"], "r") as f:
    entity_meta: Dict[str, Any] = json.load(f)

print(f"✓ Loaded entity context, vocab, and meta")
print(f"  • Total entities: {len(entity_vocab)}")

# Load image database if available
image_db_path = Path("data/entities/image_db.json")
if image_db_path.exists():
    with open(image_db_path, "r") as f:
        image_db = json.load(f)
    print(f"✓ Loaded image database: {len(image_db)} images")
else:
    image_db = {}
    print("⚠ No image_db.json found; will show image IDs only")

Loading entity graph from: /kaggle/input/flickr30k/data/graph/entity_graph.pt

  Loading graph from: /kaggle/input/flickr30k/data/graph/entity_graph.pt
  ✓ Loaded graph:
    entity nodes: 12872
    semantic edges: 293458
    co-occurrence edges: 812387
✓ Entity graph loaded
  • Entities: 12872
  • Semantic edges: 293458
  • Co-occurrence edges: 812387

Loading entity metadata...
✓ Loaded entity context, vocab, and meta
  • Total entities: 12872
⚠ No image_db.json found; will show image IDs only


## Helper Functions for Pretty Printing

These functions transform technical outputs into human-readable tables and summaries.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from PIL import Image as PILImage


def get_image_path(image_id: str) -> Path | None:
    """
    Resolve an image file path for a given image_id.
    
    Priority:
    1) If image_db has an explicit filename/path, use that under IMAGE_ROOT.
    2) Otherwise, assume image_id is a bare filename without extension and try common patterns.
    
    Args:
        image_id: Image identifier (e.g., "1000092795" or "1000092795.jpg")
    
    Returns:
        Path to the image file if found, None otherwise
    """
    # 1) Try image_db if available
    if image_db:
        info = image_db.get(image_id)
        if info is not None:
            # Try keys like 'filename' or 'path' depending on how image_db was built
            fname = info.get("filename") or info.get("path")
            if fname is not None:
                p = IMAGE_ROOT / fname
                if p.exists():
                    return p
    
    # 2) Fallback: image_id itself might already be a filename like "1000092795.jpg"
    # Try as-is under IMAGE_ROOT
    p = IMAGE_ROOT / image_id
    if p.exists():
        return p
    
    # 3) If image_id looks like "1000092795" without extension, try adding ".jpg"
    p_jpg = IMAGE_ROOT / f"{image_id}.jpg"
    if p_jpg.exists():
        return p_jpg
    
    # Could not find a file
    return None


def show_images_in_row(image_ids_scores, title: str = "", max_images: int = 5):
    """
    Display images in a single HTML row with scores.
    
    Args:
        image_ids_scores: List of (image_id, score) tuples or dict items
        title: Optional title for the image row
        max_images: Maximum number of images to display
    """
    rows = []
    for idx, (img_id, score) in enumerate(image_ids_scores[:max_images]):
        img_path = get_image_path(img_id)
        if img_path is None:
            # If we can't find the file, show a placeholder box with the ID
            rows.append(
                f"<div style='display:inline-block; margin:5px; text-align:center;'>"
                f"<div style='width:160px; height:120px; border:1px solid #ccc; "
                f"display:flex; align-items:center; justify-content:center;'>"
                f"<span style='font-size:10px;'>no image<br>{img_id}</span>"
                f"</div>"
                f"<div style='font-size:10px; margin-top:4px;'>score={score:.3f}</div>"
                f"</div>"
            )
        else:
            # Note: For Kaggle notebooks, we need to use data URIs or file:// URLs
            # Let's try using file path directly
            rows.append(
                f"<div style='display:inline-block; margin:5px; text-align:center;'>"
                f"<img src='file://{img_path}' style='max-width:160px; max-height:120px; display:block;' "
                f"onerror=\"this.parentElement.innerHTML='<div style=\\'width:160px;height:120px;border:1px solid #ccc;"
                f"display:flex;align-items:center;justify-content:center;\\'>"
                f"<span style=\\'font-size:10px;\\'>error loading<br>{img_id}</span></div>'\"/>"
                f"<div style='font-size:10px; margin-top:4px;'>{img_id[:12]}<br>score={score:.3f}</div>"
                f"</div>"
            )
    
    html = "<div style='white-space:nowrap; overflow-x:auto;'>" + "".join(rows) + "</div>"
    if title:
        display(HTML(f"<h4>{title}</h4>"))
    display(HTML(html))


def minmax_normalize(scores: Dict[str, float]) -> Dict[str, float]:
    """
    Min-max normalize scores to [0, 1] range.
    
    Args:
        scores: Dictionary mapping image_id to score
    
    Returns:
        Dictionary with normalized scores in [0, 1]
    """
    if not scores:
        return {}
    
    values = list(scores.values())
    min_val = min(values)
    max_val = max(values)
    
    # Handle edge case where all scores are equal
    if max_val - min_val < 1e-8:
        return {k: 0.5 for k in scores.keys()}
    
    return {k: (v - min_val) / (max_val - min_val) for k, v in scores.items()}


def pretty_print_seeds(candidates: List[Tuple[str, float]], image_db: Dict, max_rows: int = 5) -> None:
    """
    Print top seed images in a friendly table format.
    
    Args:
        candidates: List of (image_id, score) tuples
        image_db: Dictionary mapping image_id to metadata
        max_rows: Maximum number of rows to display
    """
    print(f"\n{'Rank':<6} {'Image ID':<15} {'Score':<8} Caption")
    print("-" * 80)
    
    for rank, (img_id, score) in enumerate(candidates[:max_rows], 1):
        caption = "N/A"
        if image_db and img_id in image_db:
            captions = image_db[img_id].get("captions", [])
            if captions:
                caption = captions[0][:60] + "..." if len(captions[0]) > 60 else captions[0]
        
        print(f"{rank:<6} {img_id:<15} {score:<8.4f} {caption}")


def pretty_print_entities(enrichment_result, top_n: int = 10) -> None:
    """
    Print discovered entities (concepts) in a friendly table with entity names.

    This function is robust: it will first try to use names provided by the
    enrichment result, and if those are missing or uninformative it will
    fall back to `entity_vocab` and `entity_meta` (which are loaded earlier
    in the notebook). If none are available it will show the raw ID.

    Args:
        enrichment_result: Result object from enrich_query()
        top_n: Number of top entities to display
    """
    # Safely extract ids / scores (enrichment_result shape may vary)
    entity_ids = getattr(enrichment_result, 'entity_ids', [])[:top_n]
    raw_names = getattr(enrichment_result, 'entity_names', None)
    entity_scores = getattr(enrichment_result, 'entity_scores', [])[:top_n]

    resolved_names = []
    for i, eid in enumerate(entity_ids):
        name = None
        # 1) Prefer a name provided by the enrichment result (if present and non-empty)
        if raw_names and i < len(raw_names) and raw_names[i]:
            name = raw_names[i]

        # 2) Fallback: look up in entity_vocab (may be a dict mapping id->str or id->meta)
        if not name and 'entity_vocab' in globals() and isinstance(entity_vocab, dict):
            v = entity_vocab.get(eid)
            if isinstance(v, str):
                name = v
            elif isinstance(v, dict):
                name = v.get('name') or v.get('label') or v.get('text')

        # 3) Fallback: look up in entity_meta (may contain human-readable 'name' or 'label')
        if not name and 'entity_meta' in globals() and isinstance(entity_meta, dict):
            m = entity_meta.get(eid)
            if isinstance(m, dict):
                name = m.get('name') or m.get('label') or m.get('title')

        # 4) Final fallback: use the raw id as string
        if not name:
            name = str(eid)

        resolved_names.append(name)

    # Ensure scores list matches length
    if len(entity_scores) < len(entity_ids):
        # pad scores with zeros if missing
        entity_scores = list(entity_scores) + [0.0] * (len(entity_ids) - len(entity_scores))

    # Create DataFrame for better formatting
    df = pd.DataFrame({
        "Rank": range(1, len(entity_ids) + 1),
        "Entity ID": entity_ids,
        "Entity Name": resolved_names,
        "Score": [f"{score:.4f}" for score in entity_scores]
    })

    print("\n" + df.to_string(index=False))


def pretty_print_graph_results(kg_result, image_db: Dict, top_n: int = 5, 
                               show_normalized: bool = False) -> Dict[str, float]:
    """
    Print top images from graph search results with optional normalization.
    
    Args:
        kg_result: Result object from graph_search()
        image_db: Dictionary mapping image_id to metadata
        top_n: Number of top images to display
        show_normalized: If True, add normalized KG scores column
    
    Returns:
        Dictionary of normalized KG scores (for reuse)
    """
    kg_scores = image_scores_dict(kg_result)
    
    # Normalize scores
    kg_scores_norm = minmax_normalize(kg_scores)
    
    # Sort by raw score descending
    sorted_items = sorted(kg_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    
    if show_normalized:
        print(f"\n{'Rank':<6} {'Image ID':<15} {'KG Score':<12} {'KG (norm)':<12} Caption")
        print("-" * 100)
    else:
        print(f"\n{'Rank':<6} {'Image ID':<15} {'KG Score':<10} Caption")
        print("-" * 80)
    
    for rank, (img_id, score) in enumerate(sorted_items, 1):
        caption = "N/A"
        if image_db and img_id in image_db:
            captions = image_db[img_id].get("captions", [])
            if captions:
                caption = captions[0][:50] + "..." if len(captions[0]) > 50 else captions[0]
        
        if show_normalized:
            norm_score = kg_scores_norm.get(img_id, 0.0)
            print(f"{rank:<6} {img_id:<15} {score:<12.4f} {norm_score:<12.4f} {caption}")
        else:
            print(f"{rank:<6} {img_id:<15} {score:<10.4f} {caption}")
    
    if show_normalized:
        print("\n💡 KG (norm) scales the KG score into [0, 1] for easier comparison with CLIP scores.")
    
    return kg_scores_norm


def plot_clip_vs_kg_comparison(candidates: List[Tuple[str, float]], 
                               kg_scores_norm: Dict[str, float],
                               top_n: int = 5) -> None:
    """
    Create a bar chart comparing CLIP scores vs normalized KG scores.
    
    Args:
        candidates: List of (image_id, clip_score) tuples from Stage 1
        kg_scores_norm: Dictionary of normalized KG scores
        top_n: Number of top images to compare
    """
    # Get top N from each
    clip_dict = {img_id: score for img_id, score in candidates[:top_n]}
    
    # Find overlapping images
    overlap_ids = set(clip_dict.keys()) & set(kg_scores_norm.keys())
    
    if not overlap_ids:
        print("\n⚠️ No overlapping images between CLIP top-5 and KG results")
        return
    
    # Prepare data for plotting
    overlap_ids = sorted(overlap_ids, key=lambda x: clip_dict[x], reverse=True)
    
    clip_scores = [clip_dict[img_id] for img_id in overlap_ids]
    kg_scores = [kg_scores_norm[img_id] for img_id in overlap_ids]
    
    # Normalize CLIP scores for fair comparison
    clip_scores_norm = list(minmax_normalize({img_id: clip_dict[img_id] for img_id in overlap_ids}).values())
    
    # Create bar chart
    x = np.arange(len(overlap_ids))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(12, 5))
    bars1 = ax.bar(x - width/2, clip_scores_norm, width, label='CLIP (normalized)', color='steelblue', alpha=0.8)
    bars2 = ax.bar(x + width/2, kg_scores, width, label='KG (normalized)', color='darkorange', alpha=0.8)
    
    ax.set_xlabel('Image ID', fontsize=11)
    ax.set_ylabel('Normalized Score [0, 1]', fontsize=11)
    ax.set_title('CLIP vs Knowledge Graph Scores (Overlapping Images)', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([img_id[:10] + '..' if len(img_id) > 10 else img_id for img_id in overlap_ids], 
                       rotation=45, ha='right')
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(
        "\n💡 Blue bars show how strongly CLIP alone likes each image. "
        "Orange bars show how strongly the concept graph (KG) likes them. "
        "Images where the orange bar is high but blue is moderate are cases where "
        "the graph adds extra value."
    )


def show_config_summary(qe_cfg: Dict[str, Any], gs_cfg: Dict[str, Any]) -> None:
    """
    Display configuration settings in human-readable format.
    """
    print("\n" + "=" * 80)
    print("CONFIGURATION SUMMARY")
    print("=" * 80)
    
    print("\n📊 Query Enrichment Settings:")
    print(f"  • Number of seed images used to discover concepts: K_seed_raw = {qe_cfg['K_seed_raw']}")
    print(f"  • Number of concepts attached to the query: M_enrich = {qe_cfg['M_enrich']}")
    print(f"  • Similarity threshold for entity relevance: sigma_sim = {qe_cfg.get('sigma_sim', 'N/A')}")
    
    print("\n🔍 Graph Search Settings:")
    print(f"  • Maximum hops in the concept graph: H_max = {gs_cfg['H_max']}")
    print(f"  • Maximum nodes processed: B = {gs_cfg['B']}")
    print(f"  • Score decay per hop: decay = {gs_cfg['decay']}")
    print(f"  • Minimum edge weight: min_weight = {gs_cfg.get('min_weight', 'N/A')}")
    
    print("\n" + "=" * 80)

print("✓ Helper functions loaded (with visualization enhancements)")

✓ Helper functions loaded


In [16]:
# Display current configuration
show_config_summary(qe_cfg, gs_cfg)


CONFIGURATION SUMMARY

📊 Query Enrichment Settings:
  • Number of seed images used to discover concepts: K_seed_raw = 32
  • Number of concepts attached to the query: M_enrich = 8
  • Similarity threshold for entity relevance: sigma_sim = N/A

🔍 Graph Search Settings:
  • Maximum hops in the concept graph: H_max = 2
  • Maximum nodes processed: B = 20
  • Score decay per hop: decay = 0.85
  • Minimum edge weight: min_weight = N/A



## Walkthrough: From Query → Enrichment → Graph Search

Let's follow one example query step by step:

1. **CLIP finds example images (seeds)** — Initial semantic search results
2. **System discovers important concepts** — Entities extracted from seed images
3. **Enriched query is built** — Original query + discovered concepts
4. **Graph walk finds boosted images** — Knowledge graph traversal scores images

In [17]:
# Pick a demo query
demo_query = "a girl playing violin on stage"
print(f"\n🎯 Demo Query: \"{demo_query}\"")
print("\nLet's see how the system processes this query...")


🎯 Demo Query: "a girl playing violin on stage"

Let's see how the system processes this query...


### Step 1: CLIP-Only Seeds (Stage 1 Retrieval)

First, we use CLIP to find images that semantically match the query.  
These become our "seed" examples.

In [ ]:
K1 = 64  # Number of candidates to retrieve in Stage 1

print(f"Retrieving top {K1} seed images using CLIP...")
candidates: List[Tuple[str, float]] = hybrid_engine._stage1_retrieve(
    query=demo_query,
    k1=K1,
    show_progress=False,
)

print(f"\n✓ Found {len(candidates)} candidate images")
print("\nTop 5 example images (seeds) from CLIP:")
pretty_print_seeds(candidates, image_db, max_rows=5)

# Display images in a row
show_images_in_row(candidates, title="📸 Top CLIP seeds (images)", max_images=5)

print(
    "\n💡 These are the first images our system thinks might match your query. "
    "We will use them as examples (seeds) to discover related concepts."
)

Retrieving top 64 seed images using CLIP...

✓ Found 64 candidate images

Top 5 example images (seeds) from CLIP:

Rank   Image ID        Score    Caption
--------------------------------------------------------------------------------
1      383051932.jpg   0.3319   N/A
2      2663670442.jpg  0.3189   N/A
3      1251716308.jpg  0.3157   N/A
4      4888969156.jpg  0.3153   N/A
5      6082587848.jpg  0.3119   N/A

💡 These are the first images our system thinks might match your query. We will use them as examples (seeds) to discover related concepts.


### Step 2: Query Enrichment with Discovered Concepts

Now we analyze the seed images to discover important concepts (entities).  
The system:
- Looks at what entities appear in the seed images
- Measures how relevant each entity is to the original query
- Selects the top M entities to "enrich" the query

In [19]:
# Prepare seeds for enrichment
K_seed_raw = qe_cfg["K_seed_raw"]
seeds_for_enrichment = candidates[:K_seed_raw]

print(f"Using top {K_seed_raw} seeds for concept discovery...\n")

# Run query enrichment
enrichment_result = enrich_query(
    query=demo_query,
    seeds=seeds_for_enrichment,
    encoders=bi_encoder,
    entity_context=entity_context,
    cfg=raw_cfg,
)

print("=" * 80)
print("QUERY ENRICHMENT RESULT")
print("=" * 80)

print("\n📝 Original query:")
print(f"   {enrichment_result.original_query}")

print("\n✨ Enriched query:")
print(f"   {enrichment_result.enriched_query}")

print("\n🔑 Top discovered concepts:")
pretty_print_entities(enrichment_result, top_n=10)

print(
    "\n💡 The system looked at the example images and found concepts that appear "
    "frequently and are semantically close to your query. It then attached those "
    "concepts to your query as 'Related: [concept1, concept2, ...]'."
)

Using top 32 seeds for concept discovery...

QUERY ENRICHMENT RESULT

📝 Original query:
   a girl playing violin on stage

✨ Enriched query:
   a girl playing violin on stage. Related: entity_12470, entity_12739, entity_3865, entity_11560, entity_3647, entity_7221, entity_6779, entity_12471

🔑 Top discovered concepts:

Rank   Concept                        Score     
--------------------------------------------------------------------------------
1      entity_12470                   0.7684    
2      entity_12739                   0.7677    
3      entity_3865                    0.7487    
4      entity_11560                   0.7397    
5      entity_3647                    0.6471    
6      entity_7221                    0.5455    
7      entity_6779                    0.5269    
8      entity_12471                   0.4997    

💡 The system looked at the example images and found concepts that appear frequently and are semantically close to your query. It then attached those concept

### Step 3: Graph Search Using Seeds + Enriched Query

Now we walk the entity knowledge graph:
- Start from seed entities (found in seed images)
- Expand through connected entities (up to H_max hops)
- Score images based on which concepts they contain
- Images with strong combinations of relevant concepts get boosted

In [ ]:
print("Running graph search...\n")

# Perform graph search
kg_result = graph_search(
    query=demo_query,
    graph=graph,
    encoders=bi_encoder,
    cfg=raw_cfg,
    seeds=seeds_for_enrichment,
)

kg_scores = image_scores_dict(kg_result)

print("=" * 80)
print("GRAPH SEARCH SUMMARY")
print("=" * 80)

print(f"\n📍 Seed entities: {len(kg_result.seed_entity_ids)}")
print(f"🔍 Total entities visited: {len(kg_result.entity_ids)}")
print(f"🖼️  Images with non-zero KG score: {len(kg_scores)}")
print(f"⏱️  Runtime: {kg_result.runtime_ms:.2f} ms")

print("\n🏆 Top images by knowledge graph score:")
kg_scores_norm = pretty_print_graph_results(kg_result, image_db, top_n=5, show_normalized=True)

# Display top KG images
kg_items_sorted = sorted(kg_scores.items(), key=lambda x: x[1], reverse=True)
show_images_in_row(kg_items_sorted, title="📸 Top KG-scored images", max_images=5)

print(
    "\n💡 These images are boosted because they contain strong combinations of the "
    "discovered concepts, even if their captions don't exactly match your words. "
    "This is the power of the knowledge graph!"
)

Running graph search...

GRAPH SEARCH SUMMARY

📍 Seed entities: 10
🔍 Total entities visited: 112
🖼️  Images with non-zero KG score: 29556
⏱️  Runtime: 559.76 ms

🏆 Top images by knowledge graph score:

Rank   Image ID        KG Score   Caption
--------------------------------------------------------------------------------
1      4879964792.jpg  200.9489   N/A
2      3269087421.jpg  198.0269   N/A
3      3723541814.jpg  197.5902   N/A
4      7545811884.jpg  191.4089   N/A
5      4637052380.jpg  189.2179   N/A

💡 These images are boosted because they contain strong combinations of the discovered concepts, even if their captions don't exactly match your words. This is the power of the knowledge graph!


### Step 4: Compare CLIP-Only vs CLIP + Knowledge Graph

Let's see the difference side by side:

In [ ]:
print("\n" + "=" * 80)
print("COMPARISON: CLIP-ONLY vs CLIP + KNOWLEDGE GRAPH")
print("=" * 80)

print("\n🔵 CLIP-ONLY (Stage 1) — Top 5:")
print("   (Pure semantic similarity between query and image embeddings)")
pretty_print_seeds(candidates, image_db, max_rows=5)

print("\n🟢 CLIP + KNOWLEDGE GRAPH — Top 5:")
print("   (Boosted by concept combinations discovered from the graph)")
pretty_print_graph_results(kg_result, image_db, top_n=5, show_normalized=True)

print(
    "\n💡 Notice how the Knowledge Graph may surface different images that have "
    "relevant concepts, potentially improving recall for complex queries."
)

# Visual comparison: Bar chart
print("\n" + "─" * 80)
print("📊 VISUAL COMPARISON: CLIP vs KG Scores")
print("─" * 80)
plot_clip_vs_kg_comparison(candidates, kg_scores_norm, top_n=10)


COMPARISON: CLIP-ONLY vs CLIP + KNOWLEDGE GRAPH

🔵 CLIP-ONLY (Stage 1) — Top 5:
   (Pure semantic similarity between query and image embeddings)

Rank   Image ID        Score    Caption
--------------------------------------------------------------------------------
1      383051932.jpg   0.3319   N/A
2      2663670442.jpg  0.3189   N/A
3      1251716308.jpg  0.3157   N/A
4      4888969156.jpg  0.3153   N/A
5      6082587848.jpg  0.3119   N/A

🟢 CLIP + KNOWLEDGE GRAPH — Top 5:
   (Boosted by concept combinations discovered from the graph)

Rank   Image ID        KG Score   Caption
--------------------------------------------------------------------------------
1      4879964792.jpg  200.9489   N/A
2      3269087421.jpg  198.0269   N/A
3      3723541814.jpg  197.5902   N/A
4      7545811884.jpg  191.4089   N/A
5      4637052380.jpg  189.2179   N/A

💡 Notice how the Knowledge Graph may surface different images that have relevant concepts, potentially improving recall for complex queries

## Try Your Own Query!

Now it's your turn! Type any query (in English) and see:
1. The first example images we find
2. The concepts we discover
3. The enriched query
4. The top images scored by the concept graph

**Example queries to try:**
- "a dog running in the park"
- "children playing soccer"
- "a sunset over the ocean"
- "a person reading a book in a library"
- "a red car on a highway"

In [ ]:
def run_query_demo(query: str, k1: int = 64, top_show: int = 5) -> None:
    """
    Run the complete Phase 4 pipeline for any query.
    
    Args:
        query: Text query to search for
        k1: Number of Stage 1 candidates to retrieve
        top_show: Number of top results to display
    """
    print("\n" + "=" * 80)
    print("PHASE 4 PIPELINE DEMO")
    print("=" * 80)
    print(f"\n🎯 QUERY: \"{query}\"")
    print("=" * 80)

    # ========== STEP 1: Stage 1 Seeds ==========
    print("\n" + "─" * 80)
    print("STEP 1 — Finding Example Images (Seeds) with CLIP")
    print("─" * 80)
    
    candidates = hybrid_engine._stage1_retrieve(
        query=query,
        k1=k1,
        show_progress=False,
    )
    
    print(f"\n✓ Retrieved {len(candidates)} candidates")
    print(f"\nTop {top_show} seed images:")
    pretty_print_seeds(candidates, image_db, max_rows=top_show)
    
    # Display images in a row
    show_images_in_row(candidates, title=f"📸 Top {top_show} CLIP Seed Images", max_images=top_show)

    # ========== STEP 2: Query Enrichment ==========
    print("\n" + "─" * 80)
    print("STEP 2 — Discovering Concepts & Enriching Query")
    print("─" * 80)
    
    K_seed_raw = qe_cfg["K_seed_raw"]
    seeds_for_enrichment = candidates[:K_seed_raw]
    
    print(f"\nAnalyzing top {K_seed_raw} seeds for concept discovery...")
    
    enrichment_result = enrich_query(
        query=query,
        seeds=seeds_for_enrichment,
        encoders=bi_encoder,
        entity_context=entity_context,
        cfg=raw_cfg,
    )
    
    print("\n📝 Original query:")
    print(f"   {enrichment_result.original_query}")
    
    print("\n✨ Enriched query:")
    print(f"   {enrichment_result.enriched_query}")
    
    print("\n🔑 Discovered concepts:")
    pretty_print_entities(enrichment_result, top_n=10)

    # ========== STEP 3: Graph Search ==========
    print("\n" + "─" * 80)
    print("STEP 3 — Walking the Concept Graph")
    print("─" * 80)
    
    print("\nTraversing entity knowledge graph...")
    
    kg_result = graph_search(
        query=query,
        graph=graph,
        encoders=bi_encoder,
        cfg=raw_cfg,
        seeds=seeds_for_enrichment,
    )
    
    kg_scores = image_scores_dict(kg_result)
    
    print(f"\n✓ Graph search complete")
    print(f"  • Seed entities: {len(kg_result.seed_entity_ids)}")
    print(f"  • Entities visited: {len(kg_result.entity_ids)}")
    print(f"  • Images scored: {len(kg_scores)}")
    print(f"  • Runtime: {kg_result.runtime_ms:.2f} ms")

    # ========== STEP 4: Results ==========
    print("\n" + "─" * 80)
    print("STEP 4 — Final Results")
    print("─" * 80)
    
    print(f"\n🏆 Top {top_show} images by concept graph score:")
    kg_scores_norm = pretty_print_graph_results(kg_result, image_db, top_n=top_show, show_normalized=True)
    
    # Display top KG images
    kg_items_sorted = sorted(kg_scores.items(), key=lambda x: x[1], reverse=True)
    show_images_in_row(kg_items_sorted, title=f"📸 Top {top_show} Knowledge Graph Images", max_images=top_show)
    
    # Visual comparison: Bar chart
    print("\n" + "─" * 80)
    print("📊 VISUAL COMPARISON: CLIP vs KG Scores")
    print("─" * 80)
    plot_clip_vs_kg_comparison(candidates, kg_scores_norm, top_n=10)
    
    # ========== Summary ==========
    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(
        "\n🎓 What just happened:\n"
        "   1. We started from your text query\n"
        "   2. Found example images using CLIP\n"
        "   3. Discovered important concepts from those examples\n"
        "   4. Used a graph of concepts to find images with relevant concept combinations\n"
        "   5. Scored images based on both semantic similarity AND concept co-occurrence\n"
        "\n💡 This is how knowledge graphs enhance traditional semantic search!"
    )
    print("=" * 80)

print("✓ Interactive demo function loaded (with visualizations)")

✓ Interactive demo function loaded


### Interactive Query Interface

Use the text box below to enter your query, then click "Run Demo" to see the full pipeline in action!

In [23]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create UI components
query_box = widgets.Text(
    value="a girl playing violin on stage",
    description="Query:",
    layout=widgets.Layout(width="80%"),
    style={'description_width': '60px'},
)

run_button = widgets.Button(
    description="🚀 Run Demo",
    button_style="primary",
    tooltip="Click to run the Phase 4 pipeline",
    layout=widgets.Layout(width="150px"),
)

output = widgets.Output()

# Button click handler
def on_run_clicked(b):
    with output:
        clear_output(wait=True)
        if query_box.value.strip():
            run_query_demo(query_box.value)
        else:
            print("⚠️ Please enter a query!")

run_button.on_click(on_run_clicked)

# Display UI
print("Enter your query and click the button to see the Phase 4 pipeline in action!\n")
display(query_box, run_button, output)

Enter your query and click the button to see the Phase 4 pipeline in action!



Text(value='a girl playing violin on stage', description='Query:', layout=Layout(width='80%'), style=TextStyle…

Button(button_style='primary', description='🚀 Run Demo', layout=Layout(width='150px'), style=ButtonStyle(), to…

Output()

## Optional: Mode Comparison (CLIP-only vs CLIP+KG vs Full Hybrid)

If you want to compare different search modes, run the cell below.  
This shows how results differ across:
- **clip_only**: Pure CLIP semantic search
- **clip_kg**: CLIP + Knowledge Graph enrichment
- **full**: CLIP + BLIP-2 + Knowledge Graph (if cross-encoder is available)

In [24]:
# Mode comparison (optional)
comparison_query = "a girl playing violin on stage"
modes = ["clip_only", "clip_kg", "full"]

print("\n" + "=" * 80)
print("MODE COMPARISON")
print("=" * 80)
print(f"\nQuery: \"{comparison_query}\"\n")

for mode in modes:
    print("\n" + "─" * 80)
    print(f"Mode: {mode.upper()}")
    print("─" * 80)
    
    try:
        results = hybrid_engine.text_to_image_hybrid_search(
            query=comparison_query,
            k1=64,
            k2=5,
            mode=mode,
            show_progress=False,
        )
        
        print(f"\nTop 5 results for mode '{mode}':")
        pretty_print_seeds(results, image_db, max_rows=5)
        
    except Exception as e:
        print(f"⚠️ Error running mode '{mode}': {e}")
        print("   (This mode may require additional components not yet configured)")

print("\n" + "=" * 80)
print(
    "💡 Notice how different modes may surface different images.\n"
    "   - 'clip_only': Fast, semantic similarity only\n"
    "   - 'clip_kg': Enhanced with knowledge graph concept boosting\n"
    "   - 'full': Maximum quality with CLIP + BLIP-2 cross-encoder + KG"
)
print("=" * 80)


MODE COMPARISON

Query: "a girl playing violin on stage"


────────────────────────────────────────────────────────────────────────────────
Mode: CLIP_ONLY
────────────────────────────────────────────────────────────────────────────────

Top 5 results for mode 'clip_only':

Rank   Image ID        Score    Caption
--------------------------------------------------------------------------------
1      383051932.jpg   0.3319   N/A
2      2663670442.jpg  0.3189   N/A
3      1251716308.jpg  0.3157   N/A
4      4888969156.jpg  0.3153   N/A
5      6082587848.jpg  0.3119   N/A

────────────────────────────────────────────────────────────────────────────────
Mode: CLIP_KG
────────────────────────────────────────────────────────────────────────────────

Top 5 results for mode 'clip_kg':

Rank   Image ID        Score    Caption
--------------------------------------------------------------------------------
1      383051932.jpg   1.0000   N/A
2      2663670442.jpg  0.7598   N/A
3      1251716308

## Conclusion

🎉 **Congratulations!** You've explored the Phase 4 pipeline:

✅ **Query Processing** — From simple text to enriched queries  
✅ **Concept Discovery** — Automatic entity extraction from seed images  
✅ **Graph-Enhanced Search** — Leveraging knowledge graphs for better retrieval  
✅ **Interactive Exploration** — Try your own queries and see results instantly  

### Key Takeaways

1. **Knowledge graphs enhance semantic search** by understanding concept relationships
2. **Query enrichment** helps bridge the semantic gap between user intent and image content
3. **Multi-hop graph traversal** discovers relevant images through concept co-occurrence
4. **Hybrid approaches** combine multiple signals (CLIP, BLIP-2, KG) for robust retrieval

### Next Steps

- Explore other notebooks: `08_hybrid_search_evaluation.ipynb`, `09_inspect_entity_graph.ipynb`
- Tune configuration parameters in `configs/entity_graph.yaml`
- Experiment with different queries and analyze the enrichment results
- Try building your own entity graph with custom entities

---

**Questions or feedback?** Check the project README or open an issue on GitHub!